## Notebook16a

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
country = pl.read_csv(ub + "data/countries.csv")

**Research Question(s)**: How much does the choice of a map projection effect the measurement of polygon areas? How distorted are the perceptions that people may have if their mental image of the world comes from popular map projections such as Web Mercator (used in most online interactive maps, including ours)?

### US State Regions: Projections

Let's start by looking at the United States before moving to data from the entire world. We can read the state's dataset that we used in the notes using the following code.

In [ ]:
state = DSGeo.read_file(ub + "data/acs_state.geojson")
state

Create an interactive plot of the state data, using the state `abb` codes as labels on the map. Zoom in and out a bit and notice how this should be similar to using something like Google Maps.

In [ ]:
DSGeo.explore(state, tooltip=[c.name])

Now, create a static plot of the state data using the coordinate system with CRS code equal to 3857. Verify that this looks (very) similar to the interactive plot in terms of the relative placement of each state and the relative size of Alaska (in particular) to the rest of the country.

In [ ]:
DSGeo.plot(state, crs=3857)

Now, let's try some other projects (using static plots for each). Take a moment to compare what they look like here. We'll do a quantitative analysis in the next section. To start, use the crs code 5069, a Conus Albers projection particularly suited for the contiguous United States. Notice that it has a more "rounded" look and makes the size of Alaska relative to the rest of the US smaller (and more accurate).

In [ ]:
DSGeo.plot(state, crs=5069)

Next use 6875, a projection used to be particularly accurate for Italy. Notice how the plot looks in comparison to the one above. You should see that it rotates the entire plot. If you can find a globe (or use something like Google Earth), it might help explain what's going on here.

In [ ]:
DSGeo.plot(state, crs=6875)

Let's go even farther on the other side of the world. Use the crs code 24378, designed to be centered on India. This will look particularly strange for some parts of the country.

In [ ]:
DSGeo.plot(state, crs=24378)

And, continuing around the world, let's do 27200, which corresponds to the New Zealand Map Grid. This is should look very weird.

In [ ]:
DSGeo.plot(state, crs=27200)

Finally, let's use the code "ESRI:54009" (make sure you put in quotes and use the prefix as well). This is the Mollweide projection, which is an equal-area projection of the world. It should produce relatively reasonable estimates of area for nearly every area of the world.

In [ ]:
DSGeo.plot(state, crs="ESRI:54009")

After taking a look at the project, can you see why we don't just always use the Mollweide for all analyses, even though it does a good job of computing areas everywhere?

### US State Regions: Projection Area

Next, we will actually compute the area of each of the regions in our dataset. We have a helper function to do this in `DSGeo.add_area`. We can put this directly inside of the `.with_columns` method to add a column to our dataset. Here is an example of how this works with the Mollweide projection, which we will treat as the "actual" area of each region in the following analyses.

In [ ]:
(
    state
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(state, crs="ESRI:54009")['area']
    )
)

Now, create a dataset that has the area defined by the Mollweide projection as well as the area from the projection 3857 (Web Mercator). Then, compute the relative error of the projection, treating the Mollweide projection as the correct area, using the formula for relative error: | real - estimated | / real. Sort by the relative errors and look at the states that have the highest errors.

In [ ]:
(
    state
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(state, crs="ESRI:54009")['area'],
        area_3857 = DSGeo.add_area(state, crs="3857")['area']
    )
    .with_columns(
        rel_error = (c.area - c.area_3857).abs() / c.area
    )
    .sort(c.rel_error)
)

Repeat the analysis that you have above with the 24378 projection (India). Take a careful look at the states with the highest errors and look back to the plot to see why these are the most distorted.

In [ ]:
(
    state
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(state, crs="ESRI:54009")['area'],
        area_24378 = DSGeo.add_area(state, crs="24378")['area']
    )
    .with_columns(
        rel_error = (c.area - c.area_24378).abs() / c.area
    )
    .sort(c.rel_error)
)

Now, create a visualization of the area (x axis) compared to the area defined by the projection 3857 (Web Mercator; y axis). Include both points and the abbreviations of the states using `geom_text_repel`. Also, include a line to see where the two calculations are equal with `+ geom_abline(linetype="dashed")`.

In [ ]:
(
    state
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(state, crs="ESRI:54009")['area'],
        area_3857 = DSGeo.add_area(state, crs="3857")['area']
    )
    .pipe(ggplot, aes("area", "area_3857"))
    + geom_point()
    + geom_text_repel(aes(label="abb"))
    + geom_abline(linetype="dashed")
)

### Countries

Next, we will move to a dataset of countries. Use the code below to read the GeoJSON file that has the official borders for all of the world countries.

In [ ]:
country_geo = DSGeo.read_file(ub + "data/countries_polygons.geojson")
country_geo

Create an interactive visualize of the data, using the name as the tooltip. Scroll in and out a bit to get a sense for how it works. Pay attention to a couple of things: (1) Zoom into a border; the one between Italy and France is a good exam, and compare our borders to those shown on the map, (2) look at Russia and see how the data treats parts of the country that extend across the +/- 180 degrees longitude.

In [ ]:
DSGeo.explore(country_geo, tooltip=[c.name])

Now, pick a country, and use the following code to find its region in the country dataset.

In [ ]:
(
    country.filter(c.iso == "JPN")
)

Next, look up a projection specifically designed for it using the look-up tool on https://epsg.io/. We want to create a static map of the countries using this projection, however, we won't be able to compute the entire world with this projection (countries on the other side of the globe will be so distored the algorithm will refuse the run). So instead, select only those countries that are in the same region as your country of choice. Note: You'll need a join for this.

In [ ]:
# I picked "6684", the code for the Japan Plane Rectangular projection
# The code here is just one way to do this, though I think it's one of
# the easiest. If you are surprised by .pipe here, keep in mind what it
# does: applies the function with the dataset as its first argument. We
# will often use it to apply custom functions within a chain of methods.
(
    country_geo
    .join(country.filter(c.region == "Asia"), on=c.iso)
    .pipe(DSGeo.plot, crs=6684)
)

Recreate the plot you had in the previous section of this notebook to show the visualization of the area (x axis) compared to the area defined by the projection 3857 (Web Mercator; y axis). Take a close look at the outliers.

In [ ]:
(
    country_geo
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(country_geo, crs="ESRI:54009")['area'],
        area_3857 = DSGeo.add_area(country_geo, crs="3857")['area']
    )
    .pipe(ggplot, aes("area", "area_3857"))
    + geom_point()
    + geom_text_repel(aes(label="name"))
    + geom_abline(linetype="dashed")
)

Now, compute the relative error of the two measurements (using Mollweide projection as the correct measurment). Create a plot showing the relative errors for the highest 20 errors using `geom_col`. Try to make the plot look nice.

In [ ]:
(
    country_geo
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(country_geo, crs="ESRI:54009")['area'],
        area_3857 = DSGeo.add_area(country_geo, crs="3857")['area']
    )
    .with_columns(
        rel_error = (c.area - c.area_3857).abs() / c.area
    )
    .sort(c.rel_error, descending=True)
    .head(n=30)
    .pipe(ggplot, aes("reorder(name, rel_error)", "rel_error"))
    + geom_col()
    + coord_flip()
)

Finally, compute the average relative error by subregion in the data. Use a `geom_col` layer to visualize the errors. Try to make the plot look nice.

In [ ]:
# I added several flourishes to the plot to make it look nice, not all
# were required in the question.
(
    country_geo
    .drop(c.geometry)
    .with_columns(
        area = DSGeo.add_area(country_geo, crs="ESRI:54009")['area'],
        area_3857 = DSGeo.add_area(country_geo, crs="3857")['area']
    )
    .with_columns(
        rel_error = (c.area - c.area_3857).abs() / c.area
    )
    .join(country, on=c.iso)
    .group_by(c.region, c.subregion)
    .agg(rel_error_mean = c.rel_error.mean())
    .pipe(ggplot, aes("reorder(subregion, rel_error_mean)", "rel_error_mean"))
    + geom_col(aes(fill="region"))
    + coord_flip()
    + scale_fill_cmap_d()
    + labs(x="Subregion", y="Mean Relative Error in Area", fill="Region")
)